# 거시·정책 변수 — 데이터 수집 & 전처리 파이프라인

## 개요

아파트 실거래가 예측 모델에 투입되는 **거시·정책 변수 4개**의 수집 출처, 수집 방법, 전처리 과정을 단계별로 기술합니다.

| 변수명 | 설명 | 단위 | 출처 |
|---|---|---|---|
| `base_rate` | 한국은행 기준금리 | % | 한국은행 ECOS |
| `mortgage_rate` | 예금은행 주택담보대출 신규취급액 가중평균 금리 | % | 한국은행 ECOS |
| `reb_price_idx` | 수원시 아파트 매매 실거래가격지수 (2017.11=100) | 지수 | 한국부동산원 R-ONE |
| `deal_year` | 거래 연도 정수 | 년 | 국토교통부 실거래가 (파생) |

### 핵심 설계 원칙
- 세 개의 시계열 변수는 **거래 연월(`_ym`) 기준 LEFT JOIN**으로 각 거래 행에 부착
- 200601 ~ 202412 (228개월) 완전 커버 → 결측 거의 없음
- 결측 발생 시 **3단계 안전망**: 구×연도별 중앙값 → 구별 중앙값 → 전체 중앙값

---
## 0. 라이브러리 임포트 & 경로 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

BASE     = r'C:\Users\최완우\OneDrive\Desktop\기계학습 기말 프로젝트_최한결'
RAW_DIR  = os.path.join(BASE, 'data', 'raw', 'macro')
FEAT_DIR = os.path.join(BASE, 'data', 'features')

print('경로 확인:')
print(f'  원본 데이터: {RAW_DIR}')
print(f'  피처 저장:   {FEAT_DIR}')
print(f'\n원본 파일 목록: {os.listdir(RAW_DIR)}')

---
## 1. 기준 데이터 로드

거시 변수는 **아파트 거래 데이터의 연월(`_ym`)에 JOIN**하는 방식으로 부착됩니다.  
따라서 먼저 기준이 되는 거래 데이터(ID 컬럼만)를 로드합니다.

In [ ]:
ID_COLS = ['aptNm', 'umdNm', '_gu', '_ym', 'dealYear', 'dealMonth',
           'dealAmount', 'floor', 'excluUseAr']

# env_features.parquet을 기준 데이터로 사용 (ID 컬럼 보유)
df_base  = pd.read_parquet(os.path.join(FEAT_DIR, 'env_features.parquet'))
df_macro = df_base[ID_COLS].copy()

print(f'기준 데이터 shape : {df_macro.shape}')
print(f'거래 연도 범위    : {df_macro["dealYear"].min()} ~ {df_macro["dealYear"].max()}')
print(f'_ym 범위          : {df_macro["_ym"].min()} ~ {df_macro["_ym"].max()}')
print(f'\n컬럼 목록:')
df_macro.head(3)

---
## 2. `base_rate` — 한국은행 기준금리

### 데이터 수집 방법

| 항목 | 내용 |
|---|---|
| **출처** | 한국은행 경제통계시스템 (ECOS) |
| **URL** | https://ecos.bok.or.kr |
| **통계 코드** | `722Y001 / 0101000` (기준금리) |
| **수집 주기** | 월별 |
| **수집 범위** | 2006년 1월 ~ 2024년 12월 (228개월) |
| **수집 방법** | ECOS 홈페이지 → 통계검색 → "기준금리" 검색 → 월별 다운로드 |

### 변수 설명
한국은행 금융통화위원회가 결정하는 **정책금리**입니다.  
금리가 오르면 대출 부담 증가 → 부동산 수요 감소 → 가격 하락 압력으로 작용합니다.

In [ ]:
# ── 원본 데이터 로드 ──────────────────────────────────────────
df_br = pd.read_parquet(os.path.join(RAW_DIR, 'base_rates.parquet'))
df_br['ym'] = df_br['ym'].astype(str)

print(f'데이터 크기: {len(df_br)}개월')
print(f'범위: {df_br["ym"].min()} ~ {df_br["ym"].max()}')
print(f'\n컬럼:\n{df_br.dtypes.to_string()}')
print(f'\n--- 앞 5행 ---')
print(df_br.head().to_string(index=False))
print(f'\n--- 기초 통계 ---')
print(df_br['base_rate'].describe().round(2).to_string())

In [ ]:
# ── 시각화 ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

x = pd.to_datetime(df_br['ym'], format='%Y%m')
ax.plot(x, df_br['base_rate'], color='steelblue', linewidth=1.8)
ax.fill_between(x, df_br['base_rate'], alpha=0.15, color='steelblue')

# 주요 이벤트 표시
events = {
    '200812': ('금융위기', 'red'),
    '202003': ('코로나19', 'orange'),
    '202204': ('금리 인상 시작', 'red'),
}
for ym, (label, color) in events.items():
    xval = pd.to_datetime(ym, format='%Y%m')
    ax.axvline(xval, color=color, linestyle='--', alpha=0.6, linewidth=1.2)
    ax.text(xval, df_br['base_rate'].max() * 0.95, label,
            fontsize=8, color=color, ha='center')

ax.set_title('한국은행 기준금리 추이 (2006.01 ~ 2024.12)', fontsize=13, pad=10)
ax.set_ylabel('기준금리 (%)')
ax.set_xlabel('연월')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f%%'))
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print(f'\n연도별 평균 기준금리:')
print(df_br.assign(year=df_br['ym'].str[:4]).groupby('year')['base_rate'].mean().round(2).to_string())

In [ ]:
# ── 전처리: LEFT JOIN ─────────────────────────────────────────
# _ym 기준으로 거래 데이터에 기준금리 부착
df_macro = df_macro.merge(
    df_br.rename(columns={'ym': '_ym'}),
    on='_ym', how='left'
)

miss = df_macro['base_rate'].isna().sum()
print(f'JOIN 후 결측: {miss}건')
if miss > 0:
    # 3단계 결측 대체
    df_macro['base_rate'] = df_macro.groupby(['_gu', 'dealYear'])['base_rate'].transform(
        lambda x: x.fillna(x.median()))
    df_macro['base_rate'] = df_macro.groupby('_gu')['base_rate'].transform(
        lambda x: x.fillna(x.median()))
    df_macro['base_rate'] = df_macro['base_rate'].fillna(df_macro['base_rate'].median())
    print(f'결측 처리 후: {df_macro["base_rate"].isna().sum()}건')

print(f'base_rate 범위: {df_macro["base_rate"].min():.2f}% ~ {df_macro["base_rate"].max():.2f}%')
print('✅ base_rate 전처리 완료')

---
## 3. `mortgage_rate` — 주택담보대출 금리

### 데이터 수집 방법

| 항목 | 내용 |
|---|---|
| **출처** | 한국은행 경제통계시스템 (ECOS) |
| **URL** | https://ecos.bok.or.kr |
| **통계 코드** | `121Y006 / BECBLA0302` (예금은행 주택담보대출 신규취급액 가중평균금리) |
| **수집 주기** | 월별 |
| **수집 범위** | 2006년 1월 ~ 2024년 12월 (228개월) |
| **수집 방법** | ECOS 홈페이지 → 금융 → 대출금리 → 주택담보대출 → 월별 다운로드 |

### 변수 설명
**실제 은행에서 소비자가 받는 주택담보대출 금리**입니다.  
기준금리에 은행 가산금리가 더해진 값으로, 항상 기준금리보다 높습니다.  
부동산 구매 시 직접적인 자금 조달 비용을 반영하여 수요에 가장 즉각적인 영향을 미칩니다.

In [ ]:
# ── 원본 데이터 로드 ──────────────────────────────────────────
df_mr = pd.read_parquet(os.path.join(RAW_DIR, 'mortgage_rates.parquet'))
df_mr['ym'] = df_mr['ym'].astype(str)

print(f'데이터 크기: {len(df_mr)}개월')
print(f'범위: {df_mr["ym"].min()} ~ {df_mr["ym"].max()}')
print(f'\n--- 앞 5행 ---')
print(df_mr.head().to_string(index=False))
print(f'\n--- 기초 통계 ---')
print(df_mr['mortgage_rate'].describe().round(2).to_string())

In [ ]:
# ── 시각화: 기준금리 vs 주담대 금리 비교 ─────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

x = pd.to_datetime(df_mr['ym'], format='%Y%m')
x_br = pd.to_datetime(df_br['ym'], format='%Y%m')

ax.plot(x_br, df_br['base_rate'],    color='steelblue', linewidth=1.8, label='기준금리 (base_rate)')
ax.plot(x,    df_mr['mortgage_rate'], color='tomato',    linewidth=1.8, label='주담대 금리 (mortgage_rate)')

# 스프레드 (gap) 음영
ax.fill_between(x, df_br['base_rate'], df_mr['mortgage_rate'],
                alpha=0.12, color='gray', label='스프레드 (가산금리)')

ax.set_title('기준금리 vs 주택담보대출 금리 비교 (2006~2024)', fontsize=13, pad=10)
ax.set_ylabel('금리 (%)')
ax.set_xlabel('연월')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# 스프레드 통계
spread = df_mr['mortgage_rate'].values - df_br['base_rate'].values
print(f'\n금리 스프레드 (주담대 - 기준금리):')
print(f'  평균: {spread.mean():.2f}%p')
print(f'  최소: {spread.min():.2f}%p')
print(f'  최대: {spread.max():.2f}%p')

In [ ]:
# ── 전처리: LEFT JOIN ─────────────────────────────────────────
df_macro = df_macro.merge(
    df_mr.rename(columns={'ym': '_ym'}),
    on='_ym', how='left'
)

miss = df_macro['mortgage_rate'].isna().sum()
print(f'JOIN 후 결측: {miss}건')
if miss > 0:
    df_macro['mortgage_rate'] = df_macro.groupby(['_gu', 'dealYear'])['mortgage_rate'].transform(
        lambda x: x.fillna(x.median()))
    df_macro['mortgage_rate'] = df_macro.groupby('_gu')['mortgage_rate'].transform(
        lambda x: x.fillna(x.median()))
    df_macro['mortgage_rate'] = df_macro['mortgage_rate'].fillna(df_macro['mortgage_rate'].median())
    print(f'결측 처리 후: {df_macro["mortgage_rate"].isna().sum()}건')

print(f'mortgage_rate 범위: {df_macro["mortgage_rate"].min():.2f}% ~ {df_macro["mortgage_rate"].max():.2f}%')
print('✅ mortgage_rate 전처리 완료')

---
## 4. `reb_price_idx` — 수원시 아파트 매매 실거래가격지수

### 데이터 수집 방법

| 항목 | 내용 |
|---|---|
| **출처** | 한국부동산원 부동산통계정보시스템 (R-ONE) |
| **URL** | https://r-one.co.kr |
| **통계명** | 수원시 아파트 매매 실거래가격지수 |
| **기준 시점** | 2017년 11월 = 100 |
| **수집 주기** | 월별 |
| **수집 범위** | 2006년 1월 ~ 2024년 12월 (228개월) |
| **수집 방법** | R-ONE → 통계 → 실거래가격지수 → 시군구별 → 수원시 → 월별 다운로드 |

### 변수 설명
**수원시 아파트 시장의 전반적인 가격 수준**을 나타내는 지수입니다.  
개별 아파트 가격이 아닌 시장 전체의 온도를 반영하므로, 지역 시황 변수로 활용합니다.  
- 지수 상승 구간 (2017~2021): 저금리 + 수요 급증 → 시장 과열
- 지수 하락 구간 (2022~2023): 금리 인상 → 시장 조정

In [ ]:
# ── 원본 데이터 로드 ──────────────────────────────────────────
df_ri = pd.read_parquet(os.path.join(RAW_DIR, 'reb_index.parquet'))
df_ri['ym'] = df_ri['ym'].astype(str)
df_ri = df_ri.rename(columns={'reb_idx': 'reb_price_idx'})

print(f'데이터 크기: {len(df_ri)}개월')
print(f'범위: {df_ri["ym"].min()} ~ {df_ri["ym"].max()}')
print(f'\n--- 앞 5행 ---')
print(df_ri.head().to_string(index=False))
print(f'\n--- 기초 통계 ---')
print(df_ri['reb_price_idx'].describe().round(2).to_string())

In [ ]:
# ── 시각화: 가격지수 추이 ─────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(13, 4))

x_ri = pd.to_datetime(df_ri['ym'], format='%Y%m')
x_br = pd.to_datetime(df_br['ym'], format='%Y%m')

# 가격지수 (왼쪽 축)
ax1.plot(x_ri, df_ri['reb_price_idx'], color='seagreen', linewidth=2, label='매매 실거래가격지수')
ax1.fill_between(x_ri, 100, df_ri['reb_price_idx'],
                 where=df_ri['reb_price_idx'] >= 100, alpha=0.15, color='seagreen', label='기준선 이상')
ax1.fill_between(x_ri, 100, df_ri['reb_price_idx'],
                 where=df_ri['reb_price_idx'] < 100, alpha=0.15, color='tomato', label='기준선 이하')
ax1.axhline(100, color='gray', linestyle='--', linewidth=1, label='기준선 (2017.11=100)')
ax1.set_ylabel('매매 실거래가격지수', color='seagreen')
ax1.set_xlabel('연월')

# 기준금리 (오른쪽 축)
ax2 = ax1.twinx()
ax2.plot(x_br, df_br['base_rate'], color='steelblue', linewidth=1.2,
         linestyle='--', alpha=0.7, label='기준금리 (우축)')
ax2.set_ylabel('기준금리 (%)', color='steelblue')

ax1.set_title('수원시 아파트 매매 실거래가격지수 & 기준금리 (2006~2024)', fontsize=13, pad=10)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

print(f'\n연도별 평균 매매가격지수:')
print(df_ri.assign(year=df_ri['ym'].str[:4]).groupby('year')['reb_price_idx'].mean().round(1).to_string())

In [ ]:
# ── 전처리: LEFT JOIN ─────────────────────────────────────────
df_macro = df_macro.merge(
    df_ri.rename(columns={'ym': '_ym'}),
    on='_ym', how='left'
)

miss = df_macro['reb_price_idx'].isna().sum()
print(f'JOIN 후 결측: {miss}건')
if miss > 0:
    df_macro['reb_price_idx'] = df_macro.groupby(['_gu', 'dealYear'])['reb_price_idx'].transform(
        lambda x: x.fillna(x.median()))
    df_macro['reb_price_idx'] = df_macro.groupby('_gu')['reb_price_idx'].transform(
        lambda x: x.fillna(x.median()))
    df_macro['reb_price_idx'] = df_macro['reb_price_idx'].fillna(df_macro['reb_price_idx'].median())
    print(f'결측 처리 후: {df_macro["reb_price_idx"].isna().sum()}건')

print(f'reb_price_idx 범위: {df_macro["reb_price_idx"].min():.1f} ~ {df_macro["reb_price_idx"].max():.1f}')
print('✅ reb_price_idx 전처리 완료')

---
## 5. `deal_year` — 거래 연도 (정수 변환)

### 설명

이 변수는 **별도 데이터 수집 없이** 원본 실거래 데이터에서 파생됩니다.

| 항목 | 내용 |
|---|---|
| **원본 컬럼** | `dealYear` (문자열, 예: "2006") |
| **변환 방법** | `int` 형 변환만 수행 |
| **출처** | 국토교통부 실거래가 공개시스템 |
| **범위** | 2006 ~ 2024 |

### 왜 별도 변수로 만드는가?
- `dealYear`는 문자열(str)이라 모델에 직접 투입 불가
- `deal_year`(int)로 변환해야 **연속형 수치 피처**로 학습 가능
- **장기 가격 트렌드** 포착: "2010년대보다 2020년대가 전반적으로 비쌈" 등의 추세를 학습

In [ ]:
# dealYear(str) → deal_year(int) 변환
df_macro['deal_year'] = df_macro['dealYear'].astype(int)

print(f'dealYear 타입: {df_macro["dealYear"].dtype}  →  deal_year 타입: {df_macro["deal_year"].dtype}')
print(f'deal_year 범위: {df_macro["deal_year"].min()} ~ {df_macro["deal_year"].max()}')
print(f'결측: {df_macro["deal_year"].isna().sum()}건')

# 연도별 거래 건수 확인
print(f'\n연도별 거래 건수:')
print(df_macro.groupby('deal_year').size().to_string())
print('\n✅ deal_year 전처리 완료')

---
## 6. 최종 검증 & 저장

4개 변수가 모두 준비되었습니다. 저장 전 결측 여부와 분포를 최종 확인합니다.

In [ ]:
MACRO_FEATURES = ['base_rate', 'mortgage_rate', 'reb_price_idx', 'deal_year']

print('=' * 50)
print('  거시·정책 4개 변수 최종 결측 확인')
print('=' * 50)
for col in MACRO_FEATURES:
    n = df_macro[col].isna().sum()
    pct = n / len(df_macro) * 100
    status = '✅' if n == 0 else '⚠️'
    print(f'  {status} {col:<20}: {n}건 ({pct:.1f}%)')

print(f'\n최종 데이터 shape: {df_macro.shape}')
print(f'\n--- 4개 변수 기초 통계 ---')
df_macro[MACRO_FEATURES].describe().round(2)

In [ ]:
# ── 4개 변수 분포 시각화 ─────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

info = {
    'base_rate':      ('기준금리 (%)',       'steelblue'),
    'mortgage_rate':  ('주담대 금리 (%)',    'tomato'),
    'reb_price_idx':  ('매매가격지수',       'seagreen'),
    'deal_year':      ('거래 연도',          'mediumpurple'),
}

for ax, (col, (label, color)) in zip(axes, info.items()):
    data = df_macro[col].dropna()
    ax.hist(data, bins=30, color=color, alpha=0.75, edgecolor='white')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel(col)
    ax.set_ylabel('건수')
    ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.2, label=f'평균: {data.mean():.2f}')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('거시·정책 변수 분포', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 저장 ──────────────────────────────────────────────────────
out_path = os.path.join(FEAT_DIR, 'macro_features.parquet')
df_macro.to_parquet(out_path, index=False)

print(f'저장 완료: {out_path}')
print(f'shape    : {df_macro.shape}')
print(f'컬럼     : {df_macro.columns.tolist()}')

# 저장 확인
df_check = pd.read_parquet(out_path)
print(f'\n[저장 파일 검증]')
print(f'  로드 shape: {df_check.shape}')
print(f'  거시 변수 결측 없음: {df_check[MACRO_FEATURES].isna().sum().sum() == 0}')
print('\n✅ macro_features.parquet 저장 완료')

---
## 7. 전체 파이프라인 요약

```
[원본 실거래 데이터]
        │
        ▼  ID 컬럼만 추출 (_ym 포함)
[기준 DataFrame]
        │
        ├── LEFT JOIN ← base_rates.parquet      (한국은행 ECOS 722Y001/0101000)
        │                → base_rate (기준금리 %)
        │
        ├── LEFT JOIN ← mortgage_rates.parquet  (한국은행 ECOS 121Y006/BECBLA0302)
        │                → mortgage_rate (주담대 금리 %)
        │
        ├── LEFT JOIN ← reb_index.parquet       (한국부동산원 R-ONE 수원시)
        │                → reb_price_idx (매매 실거래가격지수)
        │
        └── 형변환    : dealYear(str) → deal_year(int)
                        → deal_year (거래 연도 연속 피처)
        │
        ▼  결측 처리 (3단계): 구×연도 중앙값 → 구 중앙값 → 전체 중앙값
        │
        ▼
[macro_features.parquet]  — 최종 출력
```

| 변수 | 수집 기관 | 역할 |
|---|---|---|
| `base_rate` | 한국은행 ECOS | 정책금리 → 시장 유동성 대리변수 |
| `mortgage_rate` | 한국은행 ECOS | 실질 대출 비용 → 수요 직접 영향 |
| `reb_price_idx` | 한국부동산원 | 수원 지역 시황 온도계 |
| `deal_year` | 원본 파생 | 장기 가격 트렌드 포착 |